In [ ]:
from deploy import NotebookDeployer, make_param_grid
import os
import glob
import itertools
import time
import asyncio

In [ ]:
run_slurm = True

if run_slurm:
    gpus = [0,1,2,3] # our nodes always have 4 gpus per node
    os.environ["PARTITIONS"] = "kisski"
    md = NotebookDeployer(gpu_list=gpus, default_backend="python-slurm")
else:
    gpus = get_available_gpus(max_utilization=20)
    md = NotebookDeployer(gpu_list=gpus, default_backend="python")

In [ ]:
raw_configs = [
    # tp size, dp size, model len, enable expert parallel, model name
    [4, 1, 2**15, False, "openai/gpt-oss-20b"],
    [4, 1, 2**15, True, "Qwen/Qwen3-235B-A22B-Instruct-2507-FP8"],
    [4, 1, 2**15, False, "openai/gpt-oss-120b"],
    [4, 1, 2**15, True, "Qwen/Qwen3-Next-80B-A3B-Instruct-FP8"],
    [4, 1, 2**15, True, "Qwen/Qwen3-Next-80B-A3B-Thinking-FP8"],
    [4, 1, 2**15, False, "Qwen/Qwen3-4B-Thinking-2507-FP8"],
    [4, 1, 2**15, False, "zai-org/GLM-Z1-32B-0414"],
    [4, 1, 2**15, False, "zai-org/GLM-4-32B-0414"],
    [4, 1, 2**15, True, "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"],
    [4, 1, 2**15, True, "Qwen/Qwen3-30B-A3B-Thinking-2507-FP8"],
    [4, 1, 2**14, False, "RedHatAI/Llama-3.3-70B-Instruct-FP8-dynamic"],
    [2, 2, 2**14, False, "microsoft/phi-4"],
    [4, 1, 2**14, False, "mistralai/Mistral-Small-24B-Instruct-2501"],
    [4, 1, 2**14, False, "meta-llama/Llama-3.1-8B-Instruct"],
]

model_configs = {
    x[4]: {
        "tensor_parallel_size": x[0],
        "data_parallel_size": x[1],
        "max_model_len": x[2],
        "enable_expert_parallel": x[3],
        "enable_prefix_caching": True,
    }
    for x in raw_configs
}

In [ ]:
# deploy all models on all datasets

for data_dir in ["data/"]:
    for model_name, mc in list(model_configs.items()):
        logdir = f"runs_{data_dir.split('_')[-1][:-1]}/{model_name.replace('/','_')}/"

        os.makedirs(logdir, exist_ok=True)
        t = md.enqueue(notebook="run_requests.ipynb", logdir=logdir, num_gpus=4, overwrite_logdir=True,
                       model_name=model_name, model_kwargs=mc, data_dir=data_dir)
        await asyncio.sleep(0.1)

In [ ]:
# check for failed requests
for data_dir in ["data"]:
    datasets =[x.split(".jsonl")[0] for x in os.listdir(data_dir) if x.endswith(".jsonl")]
    for dataset in datasets:
        for model_name, mc in list(model_configs.items()):
            
            fn = f"{data_dir}{dataset}/{model_name.replace('/','_')}/batches_output.jsonl"
            if not os.path.exists(fn):
                print(fn)

# Merge results
***
In case youre dataset files are distributed across multiple folders, we merge them into a single "merged" directory

In [ ]:
import os

data_dirs = ["data"]
target_file = "batches_output.jsonl"

# Step 1: collect all relative paths where target_file exists
paths_by_dir = {}
for d in data_dirs:
    paths = set()
    for root, _, files in os.walk(d):
        if target_file in files:
            rel_path = os.path.relpath(root, d)
            paths.add(rel_path)
    paths_by_dir[d] = paths

# Step 2: compute intersection and warn about missing ones
all_subdirs = set().union(*paths_by_dir.values())

# Step 3: merge all, but warn when missing
for subdir in sorted(all_subdirs):
    missing = [d for d, paths in paths_by_dir.items() if subdir not in paths]
    if missing:
        print(f"⚠️ Warning: subdir '{subdir}' missing in: {', '.join(missing)}")

    # Create output dir
    out_dir = os.path.join("merged", subdir)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, target_file)

    with open(out_path, "w", encoding="utf-8") as outfile:
        for d in data_dirs:
            in_path = os.path.join(d, subdir, target_file)
            if os.path.exists(in_path):
                with open(in_path, "r", encoding="utf-8") as infile:
                    for line in infile:
                        outfile.write(line.strip() + "\n")

print("✅ Merge complete.")